In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/sanikaannadate/pc-project-files/data_background_1.npz
/kaggle/input/datasets/sanikaannadate/pc-project-files/data_background_3.npz
/kaggle/input/datasets/sanikaannadate/pc-project-files/data_background_5.npz
/kaggle/input/datasets/sanikaannadate/pc-project-files/data_cutout_1.npz
/kaggle/input/datasets/sanikaannadate/pc-project-files/data_cutout_3.npz
/kaggle/input/datasets/sanikaannadate/pc-project-files/data_cutout_5.npz
/kaggle/input/datasets/sanikaannadate/pc-project-files/data_density_1.npz
/kaggle/input/datasets/sanikaannadate/pc-project-files/data_density_3.npz
/kaggle/input/datasets/sanikaannadate/pc-project-files/data_density_5.npz
/kaggle/input/datasets/sanikaannadate/pc-project-files/data_density_inc_1.npz
/kaggle/input/datasets/sanikaannadate/pc-project-files/data_density_inc_3.npz
/kaggle/input/datasets/sanikaannadate/pc-project-files/data_density_inc_5.npz
/kaggle/input/datasets/sanikaannadate/pc-project-files/data_distortion_1.npz
/kaggle/input/dat

In [3]:
import sys, os, glob, shutil, time
if not os.path.exists('/tmp/dgcnn'):
    !git clone -q https://github.com/WangYueFt/dgcnn.git /tmp/dgcnn
if not os.path.exists('/tmp/modelnet40_c'):
    !wget -q -O /tmp/mn40c.zip "https://zenodo.org/records/6017834/files/modelnet40_c.zip?download=1"
    !unzip -qn /tmp/mn40c.zip -d /tmp/ && rm /tmp/mn40c.zip
sys.path.insert(0, '/tmp/dgcnn/pytorch')
C = '/tmp/modelnet40_c'
print(len(os.listdir(C)), 'items in ModelNet40-C')
print(glob.glob('/kaggle/input/**/*.t7', recursive=True))

78 items in ModelNet40-C
['/kaggle/input/datasets/sanikaannadate/pc-project-files/dgcnn_model.t7', '/kaggle/input/datasets/sanikaannadate/pc-project-files/pointnet_model.t7']


In [2]:
OUT_ROOT = '/kaggle/working/results'
os.makedirs(f'{OUT_ROOT}/dgcnn', exist_ok=True)
for p in glob.glob('/kaggle/input/**/*.npz', recursive=True):
    shutil.copy(p, f'{OUT_ROOT}/dgcnn/')
print(len(os.listdir(f'{OUT_ROOT}/dgcnn')), 'DGCNN files already done')

16 DGCNN files already done


In [4]:
import torch, torch.nn as nn, torch.nn.functional as F, numpy as np
from model import PointNet, DGCNN
class A: k=20; emb_dims=1024; dropout=0.5
CKPT = {os.path.basename(p): p for p in glob.glob('/kaggle/input/**/*.t7', recursive=True)}
labels = np.load(f'{C}/label.npy').flatten()

def load_model(name):
    net = DGCNN(A(), output_channels=40) if name == 'dgcnn' else PointNet(A(), output_channels=40)
    m = nn.DataParallel(net).cuda()
    m.load_state_dict(torch.load(CKPT[f'{name}_model.t7'], map_location='cuda'))
    return m.eval()

def predict(pts, bs=128):
    out = []
    with torch.no_grad():
        for i in range(0, len(pts), bs):
            x = torch.from_numpy(pts[i:i+bs]).float().cuda().permute(0, 2, 1)
            out.append(F.softmax(model(x), 1).cpu())
    return torch.cat(out).numpy()

def perturb(pts, sigma=0.01, drop=0.1):
    p = pts + np.clip(sigma*np.random.randn(*pts.shape), -2*sigma, 2*sigma)
    for b in range(len(p)):
        p[b, np.random.rand(p.shape[1]) < drop] = p[b, 0]
    return p.astype(np.float32)

def dropout_on(on):
    for m in model.modules():
        if isinstance(m, nn.Dropout): m.train(on)

def run(file, K=8):
    pts = np.load(f'{C}/{file}').astype(np.float32)
    probs = predict(pts); pred = probs.argmax(1); idx = np.arange(len(pred))
    correct = (pred == labels).astype(int)
    msp = probs.max(1)
    neg_entropy = (probs * np.log(probs + 1e-12)).sum(1)
    pert = np.stack([predict(perturb(pts)) for _ in range(K)])
    ours = pert[:, idx, pred].mean(0)
    ours_agree = (pert.argmax(2) == pred[None]).mean(0)
    dropout_on(True)
    mc = np.stack([predict(pts) for _ in range(K)]); dropout_on(False)
    mcd = mc[:, idx, pred].mean(0)
    mcd_agree = (mc.argmax(2) == pred[None]).mean(0)
    return dict(correct=correct, msp=msp, entropy=neg_entropy, ours=ours, ours_agree=ours_agree,
                mcd=mcd, mcd_agree=mcd_agree, avg_pred=pert.mean(0).argmax(1))

In [ ]:
#running PointNet

OUT_ROOT = '/kaggle/working/results'
def run_all(name, severities=None):
    global model
    model = load_model(name)
    out = f'{OUT_ROOT}/{name}'; os.makedirs(out, exist_ok=True)
    files = sorted(f for f in os.listdir(C) if f.startswith('data_'))
    if severities:
        files = [f for f in files if f == 'data_original.npy' or any(f.endswith(f'_{s}.npy') for s in severities)]
    print(len(files), 'files', flush=True)
    for i, f in enumerate(files):
        p = f'{out}/{f[:-4]}.npz'
        if os.path.exists(p): continue
        t = time.time(); r = run(f); np.savez(p, **r)
        print(name, f'{i+1}/{len(files)}', f, round(r['correct'].mean(), 3), f'{round(time.time()-t)}s', flush=True)

run_all('pointnet', severities=[1, 3, 5])

46 files
pointnet 1/46 data_background_1.npy 0.131 16s
pointnet 2/46 data_background_3.npy 0.112 10s
